# Event Impact Analysis

Effect of public holidays, events and event size on `arrival_delay`.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.events as an

TRAIN, TEST, lf = setup_analysis("03_analysis_6-events")
lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])

# has_event is stored incorrectly in parquet — recompute here
lf_all = lf_all.with_columns(
    (pl.col("event_name").cast(pl.Utf8) != "no_event").alias("has_event")
)
lf_delay = lf_all.filter(pl.col("canceled") == False)

%load_ext autoreload
%autoreload 2

## Holidays

Public holidays vs normal days — does reduced traffic improve or worsen punctuality?

In [ ]:
an.plot_events_overview(lf_delay, cfg)

In [ ]:
show_df(an.table_events_overview(lf_delay))

**Beobachtung:** Klares Ergebnis — aber mit einer Überraschung.

**Ø Delay nach Tages-Kategorie:**
| Kategorie | Ø Delay (s) | OTP | N |
|:---|---:|---:|---:|
| **Feiertag** | **46.3** | **90.6%** | 2.3M |
| Normal | 56.2 | 87.0% | 70.5M |
| Event klein (1) | **56.2** | 86.9% | 8.7M |
| Event mittel (2) | 58.9 | 86.4% | 7.5M |
| **Event gross (3)** | **66.7** | **82.4%** | 724k |

**Feiertagseffekt gegenläufig zur Erwartung:** Feiertage zeigen deutlich *weniger* Verspätung als Normaltage (46.3s vs. 56.2s, −9.9s, OTP +3.6pp). Weniger Berufsverkehr überwiegt den Freizeitverkehr an Feiertagen.

**Event-Skalierung bestätigt:** Gross-Events (+10.5s über Normal) haben klare Auswirkungen. Mittel-Events (+2.7s) sind messbar aber moderat. **Kleine Events (+0.05s) sind praktisch nicht von Normal zu unterscheiden** — Event-Gewicht 1 hat kaum Vorhersagekraft.

→ `is_holiday` als starkes Feature (−9.9s Effekt). `event_weight` als ordinales Feature; Klasse 1 eventuell binarisieren. Events sind selten (n=724k Gross, n=16.9M aller Events vs. 70.5M Normal) — Unbalanced-Class-Problem beachten.

## Event-Typ + Stunden-Profil

Welche Veranstaltungstypen haben den grössten Einfluss? Und: Wann schlägt der Effekt durch — zu welcher Stunde sind Event-Tage deutlich schlechter als Normaltage?

In [ ]:
an.plot_event_type_hourly_profile(lf_delay, cfg)

**Beobachtung:** Das Stunden-Profil bestätigt: Event-Tage sind **abends (18–22h) deutlich schlechter** als Normaltage, tagsüber kaum unterschiedlich. Das erklärt den 21h-Spike im Temporal-Profil (F-TEMP-01): nicht Kaskaden, sondern Events-Abreisewellen.

**Event-Typ-Ranking:**
| Event-Typ | Ø Delay (s) | OTP | N |
|:---|---:|---:|---:|
| **Fachmesse** | **66.0** | 84% | 4.3M |
| Konzert | 61.4 | 85% | 403k |
| Schweizer Cup | 58.1 | 87% | 348k |
| Stadtfest | 57.6 | 86% | 859k |
| Kongress | 57.4 | 86% | 2.5M |
| Super League | 53.8 | 88% | 8.2M |
| Feiertag | 46.3 | 91% | 2.3M |

**Kernbefund:** **Fachmessen** (66.0s, OTP 84%) sind die problematischste Veranstaltungskategorie — nicht Konzerte wie ursprünglich angenommen. Fachmessen (Züspa, Art Zürich, diverse Messe-Zürich-Events) dauern ganztags und ziehen über mehrere Tage grossen Verkehr auf Linie 11 (Messe Zürich / Hallenstadion Korridor). Konzerte (61.4s) sind punktueller aber intensiver (kurze Abreisewelle nach Konzertende).

**Super League** (53.8s, 8.2M Halte) — viele Beobachtungen aber nahe Normal — Fussballspiele verteilen sich besser über den Tag (lange Anfahrt, Fankurven früh präsent).

→ `event_type` als kategorisches Feature; `is_holiday` als stärkstes negatives Signal (46.3s); Fach messen-Effekt besonders für L11-Prognose relevant.

In [ ]:
an.plot_event_district_effect(lf_delay, cfg)
show_df(an.table_event_district_effect(lf_delay))

## Event-Standorte — Stadtkreise

**Beobachtung:** Der Event-Effekt auf Stadtkreis-Ebene ist überraschend klein — und in mehreren Kreisen sogar negativ.

**Δ Delay Event-Tag vs. Normaltag nach Stadtkreis:**
| Stadtkreis | Normal (s) | Event-Tag (s) | Δ |
|:---|---:|---:|---:|
| Kreis 2 | 56.1 | 59.1 | **+3.0** |
| Kreis 9 | 59.1 | 61.5 | **+2.4** |
| Kreis 3 | 55.7 | 56.9 | +1.2 |
| Kreis 4 | 54.5 | 55.7 | +1.2 |
| Kreis 11 | 68.5 | 67.7 | **−0.8** (leicht besser!) |
| Kreis 5 | 50.2 | 48.9 | −1.2 |
| outside | 58.8 | 57.2 | −1.6 |

**Kernbefund:** Nur Kreise 2 (+3.0s) und 9 (+2.4s) zeigen messbare positive Event-Effekte. Kreis 11 (wo Hallenstadion und Messe Zürich liegen!) zeigt an Event-Tagen sogar leicht *niedrigere* Delays (−0.8s) — überraschend.

**Erklärungsansatz:** Die Event-Klassifizierung ist netzweit (ganzer Betriebstag), nicht linienbezogen. Der starke Effekt einer Fachmesse auf L11-Abendstunden wird durch den gesamten Tagesbetrieb von Kreis 11 "verdünnt". Ein Kreis-Δ von +3.0s entspricht einem sehr kleinen Effekt relativ zur Kreis-Streuung — die räumliche Aggregation verbirgt den zeitlichen (Abend-)Effekt.

→ Feature-Empfehlung: `has_event × hour` Interaktion ist aussagekräftiger als nur `district × has_event`. Der Abend-Effekt (F-EVNT-03) bleibt die stärkste räumlich-zeitliche Signatur.

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-EVNT-01 | Feiertagseffekt gegenläufig: Feiertage 46.3s vs. Normal 56.2s (−9.9s, OTP +3.6pp) — Berufsverkehrsreduktion überwiegt klar | done |
| F-EVNT-02 | Event-Skalierung bestätigt: Gross +10.5s (66.7s), Mittel +2.7s, **Klein ≈ +0.05s (=Normal)**. `event_weight` ist ordinales Feature; Klasse 1 hat kaum Vorhersagekraft. | done |
| F-EVNT-03 | Event-Effekt ist primär **Abend-Phänomen (18–22h)** — tagsüber kein Unterschied. Erklärt den 21h-Spike aus F-TEMP-01. | done |
| F-EVNT-04 | **Fachmessen** schlechteste Kategorie (66.0s, OTP 84%) — nicht Konzerte. Fachmessen dauern ganztags über mehrere Tage (Messe Zürich / L11-Korridor). Konzerte 61.4s, Super League 53.8s nahe Normal. | done |
| F-EVNT-05 | Events selten: Gross n=724k vs. Normal 70.5M — stark unbalanced. `is_holiday` als stärkstes einzelnes Event-Feature (−9.9s Effekt). | done |
| F-EVNT-06 | Stadtkreis-Δ auf Event-Tagen minimal (max +3.0s in Kreis 2). Kreis 11 (Hallenstadion-Korridor) überraschend leicht besser (−0.8s) — räumliche Aggregation verbirgt Abend-Effekt. Feature-Empfehlung: `has_event × hour` Interaktion. | done |